In [2]:
from astropy.io import fits
import matplotlib.pyplot as plt
from astropy.visualization import ImageNormalize, AsinhStretch
import numpy as np
from vis import plot_model_residual, plot_data_psf
import numpy as np
from scipy.ndimage import map_coordinates, zoom

In [3]:
## here my science cutout is at 0.05”/pix, while the PSF you want to use is on 0.063”/pixel.

data = fits.open("../data/RUBIES-BLAGN-1/F356W.fits")
psf_file1 = fits.open("../PSF_files/F356W.fits")
psf_file = "../PSF_files/F356W.fits"
out_file = "../PSF_files/RUBIES-BLAGN-1/F356W.fits"

print(abs(data[0].header.get("CD1_1"))*3600)  
print(psf_file1[1].header.get("PIXELSCL"))           # pixel_scale = abs(-1.3888889e-05) * 3600 ≈ 0.05 arcsec/pixel

0.049999999999996804
0.063


In [4]:
pix_science = abs(data[0].header.get("CD1_1"))*3600
pix_psf = abs(psf_file1[1].header.get("PIXELSCL"))
print(pix_science)
print(pix_psf)

zoom_factor = pix_psf / pix_science   

with fits.open(psf_file) as hdul:
    psf = hdul[1].data.astype(float)   # DET_SAMP extension
    hdr = hdul[1].header.copy()


psf_resampled = zoom(psf, zoom_factor, order=3)
psf_resampled /= np.sum(psf_resampled)          # Renormalize total flux

newhdr = fits.Header()
newhdr["PIXELSCL"] = pix_science
newhdr["COMMENT"] = "Resampled from DET_SAMP extension"

fits.PrimaryHDU(psf_resampled, header=newhdr).writeto(out_file, overwrite=True) ### ADD THE PATH HERE

print('done')

0.049999999999996804
0.063
done


In [5]:
## check
## here my science cutout is at 0.05”/pix, while the PSF you want to use is on 0.063”/pixel.

a = fits.open("../data/J0100-2017/F356W.fits")
b_psf = fits.open("../PSF_files/J0100-2017/F356W.fits")

print(abs(a[0].header.get("CD1_1"))*3600)  
print(b_psf[0].header.get("PIXELSCL"))           # pixel_scale = abs(-1.3888889e-05) * 3600 ≈ 0.05 arcsec/pixel

0.049999999999996804
0.0499999999999968


In [6]:
### TESTS
print("Original shape:", psf.shape)
print("Resampled shape:", psf_resampled.shape)
print("Sum:", psf_resampled.sum())

ypeak, xpeak = np.unravel_index(np.argmax(psf_resampled), psf_resampled.shape)
print("Peak pixel:", xpeak, ypeak)

Original shape: (159, 159)
Resampled shape: (200, 200)
Sum: 1.0
Peak pixel: 99 100
